# Chapter 11 — nanoGPT: Attention from First Principles
**Math for ML Engineers** · companion notebook

We build the decoder-only Transformer in NumPy first — to see each equation — then run the full PyTorch nanoGPT on a tiny corpus.

**Equations implemented:** SDPA (11.1), causal mask (11.29), MHA (11.11), RoPE rotation (11.14), cross-entropy LM objective (3.33).

⚠️  The PyTorch section (cell 9+) trains faster on GPU — Colab free tier works fine.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Scaled dot-product attention (SDPA)

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

The $1/\sqrt{d_k}$ scale is **not** optional: without it, dot products of random unit vectors grow as $\sqrt{d_k}$, pushing the softmax into a near-one-hot regime where gradients vanish.  This is the derivation in §11.2.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def sdpa(Q, K, V, mask=None):
    '''Scaled dot-product attention.  Q,K,V: (seq, d_k).'''
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)           # (n, n)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)  # -inf on masked positions
    A = softmax(scores)                        # attention weights
    return A @ V, A

# Quick sanity check
rng = np.random.default_rng(0)
n, d_k = 6, 16
Q = rng.standard_normal((n, d_k))
K = rng.standard_normal((n, d_k))
V = rng.standard_normal((n, d_k))
out, A = sdpa(Q, K, V)
print(f'Output shape: {out.shape}  Attention row sums: {A.sum(axis=-1).round(4)}')


## Causal mask

For autoregressive generation, position $i$ may only attend to positions $j \leq i$ (the lower triangle including the diagonal).  The causal mask is a boolean matrix:

$$M_{ij} = \begin{cases} 1 & j \leq i \\ 0 & j > i \end{cases}$$

Inverting this (masking the upper triangle) is the most common source of look-ahead leaks — see Failure Mode 4 in Appendix E.

In [ ]:
causal_mask = np.tril(np.ones((n, n), dtype=bool))  # lower triangle

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, mat, title in zip(
        axes,
        [A,                           # without mask
         sdpa(Q, K, V, mask=causal_mask)[1],  # with causal mask
         causal_mask.astype(float)],
        ['Attn (no mask)', 'Attn (causal mask)', 'Causal mask']):
    im = ax.imshow(mat, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title); ax.set_xlabel('Key pos'); ax.set_ylabel('Query pos')
plt.colorbar(im, ax=axes[-1], fraction=0.046)
plt.suptitle('Attention weights: full vs causal', y=1.02)
plt.tight_layout(); plt.show()


## RoPE positional encoding

RoPE encodes position $m$ by rotating query/key vectors in 2D planes:

$$R_m \mathbf{q} = \begin{pmatrix} q_1 \cos m\theta_1 - q_2 \sin m\theta_1 \\ q_1 \sin m\theta_1 + q_2 \cos m\theta_1 \\ \vdots \end{pmatrix}$$

The base frequency $\theta = 10{,}000$ (original) or $500{,}000$ (Llama-3) controls the extrapolation range.  Using 10 000 when the model needs 500 000 causes the positional encoding to wrap around — Failure Mode 2.

In [ ]:
def rope_rotate(x, positions, theta_base=10000):
    '''Apply RoPE to x: (n, d).  Returns rotated tensor of same shape.'''
    n, d = x.shape
    assert d % 2 == 0
    # Frequency for each dimension pair
    i = np.arange(d // 2)
    freq = 1.0 / (theta_base ** (2*i / d))    # shape (d/2,)
    angles = np.outer(positions, freq)          # (n, d/2)
    cos, sin = np.cos(angles), np.sin(angles)
    # Reshape x into pairs
    x1, x2 = x[:, 0::2], x[:, 1::2]          # (n, d/2) each
    out = np.empty_like(x)
    out[:, 0::2] = x1*cos - x2*sin
    out[:, 1::2] = x1*sin + x2*cos
    return out

pos = np.arange(n)
Q_rope = rope_rotate(Q, pos)
print('RoPE sanity: norms preserved?', np.allclose(np.linalg.norm(Q, axis=-1),
                                                    np.linalg.norm(Q_rope, axis=-1)))
# Show how relative-position dot products decay with distance
q0 = Q_rope[0:1]  # query at position 0
dots = [(q0 @ rope_rotate(Q[k:k+1], np.array([k])).T).item() for k in range(n)]
plt.figure(figsize=(5,3))
plt.bar(range(n), dots)
plt.xlabel('Key position'); plt.ylabel('q₀ · k_pos')
plt.title('RoPE: dot product decays with relative distance')
plt.tight_layout(); plt.show()


## Full PyTorch training (nanoGPT)

The code below runs the full nanoGPT from `code/reference/nano_gpt.py`.  Expects `tinyshakespeare.txt` — download with the cell below.

**What to watch:** training loss should fall from ~4.2 (uniform over 65-char vocabulary) to ~1.5 in 2 000 steps.  Loss below 1.0 means the model has begun memorising.

In [ ]:
# Download tinyshakespeare if not present
import os
if not os.path.exists('tinyshakespeare.txt'):
    import urllib.request
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, 'tinyshakespeare.txt')
    print('Downloaded tinyshakespeare.txt')
else:
    print('tinyshakespeare.txt already present')


In [ ]:
# Copy nano_gpt.py from the book's code/reference/ and run it
# (In Colab, upload nano_gpt.py from the repo's code/reference/ directory)
# Then: exec(open('nano_gpt.py').read())
#
# Or run inline with reduced config for speed:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    print(f'PyTorch {torch.__version__}, device: {"cuda" if torch.cuda.is_available() else "cpu"}')
except ImportError:
    print('torch not available — install with: pip install torch')


## Exercise

1. In the NumPy SDPA above, set $d_k = 1$ instead of 16 and regenerate the attention heatmap.  Describe what happens to the distribution and why (§11.2, the variance argument).
2. In `rope_rotate`, try `theta_base=10000` vs `theta_base=500000`.  Plot the dot-product decay curve for each.  Which one allows the model to distinguish positions 4 000 and 8 000? Why does Llama-3 use 500 000 (§11.5.2)?